In [4]:
import os
os.chdir("../../")#与easymlops同级目录

### 拆分数据

In [5]:
import pandas as pd
data=pd.read_csv("./data/demo2.csv",encoding="gbk")
data.head(5)

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"


In [6]:
x_train=data[:5000]
x_test=data[5000:]
y_train=x_train["label"]
y_test=x_test["label"]
del x_train["label"]
del x_test["label"]

### pipe构建

In [7]:
from easymlops import NLPPipeline
from easymlops.table.preprocessing import *
from easymlops.table.ensemble import *
from easymlops.nlp.preprocessing import *
from easymlops.nlp.representation import *
from easymlops.table.perfopt import *
from easymlops.table.classification import *
from easymlops.nlp.similarity import FaissSimilarity

In [8]:
nlp=NLPPipeline()
nlp.pipe(FixInput())\
   .pipe(FillNa())\
   .pipe(SelectCols(cols=["review"]))\
   .pipe(ExtractChineseWords())\
   .pipe(ExtractJieBaWords())\
   .pipe(Parallel([LsiTopicModel(num_topics=128),Word2VecModel(embedding_size=128)]))\
   .pipe(Normalizer())\
   .pipe(FaissSimilarity(raw_data=x_train["review"].values.tolist(),create_index_param="IVF100,PQ16",create_index_measure="METRIC_INNER_PRODUCT"))

nlp.fit(x_train)

ModuleNotFoundError: No module named 'jieba'

In [ ]:
x_test_new=nlp.transform(x_test)

In [ ]:
x_test_new[:5]

In [ ]:
nlp[-1].search({"review":"这个酒店真不错，位置好，房间很干净，下次来还会选择这个酒店"},top_k=5)

In [ ]:
nlp.transform_single({"review":"这个酒店真不错，位置好，房间很干净，下次来还会选择这个酒店"})

### 性能、一致性测试

In [ ]:
nlp.auto_test(x_test[:10])

### 持久化测试

In [ ]:
nlp[-1].save_index_raw_data("similary")

In [ ]:
nlp.save("ml.pkl")

In [ ]:
nlp=NLPPipeline()
nlp.pipe(FixInput())\
   .pipe(FillNa())\
   .pipe(SelectCols(cols=["review"]))\
   .pipe(ExtractChineseWords())\
   .pipe(ExtractJieBaWords())\
   .pipe(Parallel([LsiTopicModel(num_topics=128),Word2VecModel(embedding_size=128)]))\
   .pipe(Normalizer())\
   .pipe(FaissSimilarity(raw_data=x_train["review"].values.tolist(),create_index_param="IVF100,PQ16",create_index_measure="METRIC_INNER_PRODUCT"))

In [ ]:
nlp.load("ml.pkl")

In [ ]:
nlp[-1].load_index_raw_data("similary")

In [ ]:
nlp.auto_test(x_test[:10])